# Telco Churn Analysis & Interactive Dashboard
This dashboard is designed to analyze customer churn behavior in a telecommunications company, helping stakeholders identify key factors leading to customer loss and track essential business KPIs.

### 1. Importing Required Libraries
Importing Dash for building the web application, Dash Bootstrap Components for UI styling, Plotly Express for interactive visualizations, and Pandas for data manipulation.

In [215]:
import dash
from dash import dcc, html
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd

### 2. Reading the Dataset
Loading the cleaned telco dataset from the CSV file into a Pandas DataFrame for analysis and dashboard rendering.

In [216]:
df = pd.read_csv("clean.csv")

### 3. Key Performance Indicators (KPI Cards)
Calculating and creating 6 core metric cards to display at the top of the dashboard for quick business insights:
- **Total Customers**: Overall customer base size.
- **Churn Rate**: Percentage of customers who left the service.
- **Active Customers**: Currently retained and active subscribers.
- **Avg Monthly Charges**: Average billing amount per customer.
- **Total Revenue**: Cumulative revenue generated from monthly charges.
- **New Customers**: Subscribers with a tenure of one year or less.

In [217]:
# Card1
total_customers = len(df)

kpi_card_1 = dbc.Card(
    dbc.CardBody([
        html.H5("Total Customers", className="text-muted"),
        html.H3(f"{total_customers:,}", className="text-primary") 
    ]),
    className="shadow-sm text-center mb-4" 
)

In [218]:
# Card2
churn_count = df[df['Churn Label'].astype(str).str.strip().str.lower() == 'yes'].shape[0]
churn_rate = (churn_count / total_customers) * 100

kpi_card_2 = dbc.Card(
    dbc.CardBody([
        html.H5("Churn Rate", className="text-muted"),
        html.H3(f"{churn_rate:.1f}%", className="text-danger") 
    ]),
    className="shadow-sm text-center mb-4"
)

In [219]:
# card3
kpi_card_3 = dbc.Card(
    dbc.CardBody([
        html.H5("Active Customers", className="text-muted"),
        html.H3(f"{total_customers - churn_count:,}", className="text-success")
    ]),
    className="shadow-sm text-center mb-4"
)

In [220]:
# card4
monthly_col = next((col for col in ['Monthly Charges', 'MonthlyCharges', 'monthly_charges'] if col in df.columns), df.columns[1])
avg_monthly = df[monthly_col].mean()
kpi_card_4 = dbc.Card(
    dbc.CardBody([
        html.H5("Avg Monthly Charges", className="text-muted"),
        html.H3(f"${avg_monthly:.2f}", className="text-info")
    ]),
    className="shadow-sm text-center mb-4"
)

In [221]:
# card5
monthly_col = next((col for col in ['Monthly Charges', 'MonthlyCharges', 'monthly_charges'] if col in df.columns), df.columns[1])
total_revenue = df[monthly_col].sum()
kpi_card_5 = dbc.Card(
    dbc.CardBody([
        html.H5("Total Revenue", className="text-muted"),
        html.H3(f"${total_revenue:,.0f}", className="text-warning")
    ]),
    className="shadow-sm text-center mb-4"
)

In [222]:
# card6
tenure_col = next((col for col in ['Tenure in Months', 'tenure', 'Tenure'] if col in df.columns), df.columns[0])
new_cust_count = df[df[tenure_col] <= 12].shape[0]

kpi_card_6 = dbc.Card(
    dbc.CardBody([
        html.H5("New Customers (≤1 Yr)", className="text-muted"),
        html.H3(f"{new_cust_count:,}", className="text-primary")
    ]),
    className="shadow-sm text-center mb-4"
)

### 4. Interactive Visualizations (Analytics Graphs)
Creating 6 diverse Plotly charts to explore patterns and correlations:
1. **Monthly Charges Distribution**: Highlights how higher billing amounts correlate with increased churn.
2. **Tenure by Contract Type**: Demonstrates that longer contracts significantly improve customer retention.
3. **Churn by Payment Method**: Identifies which payment methods experience higher customer drop-off.
4. **Contract Types Proportion**: Shows the overall distribution and reliance on month-to-month contracts.
5. **Tenure vs. Monthly Charges**: Scatter plot mapping customer tenure against monthly fees and churn status.
6. **Total Charges Distribution**: Violin plot illustrating the lifetime value and cumulative charges of churned vs. active users.

In [223]:
# graph1
monthly_col = next((col for col in ['Monthly Charge', 'Monthly Charges', 'MonthlyCharges', 'monthly_charges'] if col in df.columns), df.columns[1])
fig_1 = px.histogram(df, x=monthly_col, color="Churn Label", nbins=30, title="Monthly Charges Distribution")
fig_1.update_layout(margin=dict(l=20, r=20, t=40, b=20))
graph_1 = dcc.Graph(figure=fig_1)

In [224]:
# graph2
tenure_col = next((col for col in ['Tenure in Months', 'tenure', 'Tenure'] if col in df.columns), df.columns[0])
contract_col = next((col for col in df.columns if 'contract' in col.lower()), df.columns[1])
fig_2 = px.box(df, x=contract_col, y=tenure_col, color="Churn Label", title="Tenure by Contract Type")
fig_2.update_layout(margin=dict(l=20, r=20, t=40, b=20))
graph_2 = dcc.Graph(figure=fig_2)

In [225]:
# graph3
payment_col = next((col for col in df.columns if 'payment' in col.lower() or 'method' in col.lower()), df.columns[0])
fig_3 = px.histogram(df, x=payment_col, color="Churn Label", barmode="group", title="Churn by Payment Method")
fig_3.update_layout(margin=dict(l=20, r=20, t=40, b=20), xaxis_tickangle=-15)
graph_3 = dcc.Graph(figure=fig_3)

In [226]:
# graph4
contract_col = next((col for col in df.columns if 'contract' in col.lower()), df.columns[0])
fig_4 = px.pie(df, names=contract_col, title="Contract Types Proportion", hole=0.4)
fig_4.update_layout(margin=dict(l=20, r=20, t=40, b=20))
graph_4 = dcc.Graph(figure=fig_4)

In [227]:
# graph5
tenure_col = next((col for col in ['Tenure in Months', 'tenure', 'Tenure'] if col in df.columns), df.columns[0])
monthly_col = next((col for col in ['Monthly Charge', 'Monthly Charges', 'MonthlyCharges'] if col in df.columns), df.columns[1])
fig_5 = px.scatter(df, x=tenure_col, y=monthly_col, color="Churn Label", title="Tenure vs Monthly Charges", opacity=0.7)
fig_5.update_layout(margin=dict(l=20, r=20, t=40, b=20))
graph_5 = dcc.Graph(figure=fig_5)

In [228]:
# graph6
total_col = next((col for col in df.columns if 'total' in col.lower()), monthly_col)
fig_6 = px.violin(df, x="Churn Label", y=total_col, color="Churn Label", box=True, points="all", title="Total Charges Distribution by Churn")
fig_6.update_layout(margin=dict(l=20, r=20, t=40, b=20))
graph_6 = dcc.Graph(figure=fig_6)

### 5. Dashboard Layout & Architecture
Structuring the web interface using Dash Bootstrap Components (`dbc.Row` and `dbc.Col`) with a responsive grid system: placing the 6 KPI cards in the top row, followed by two clean grid rows containing the 6 analytical charts.

In [229]:
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])

app.layout = html.Div([
    html.H1("Telco Churn Dashboard", className="mb-4"),

    dbc.Row(
        [
            dbc.Col(kpi_card_1, width=2),
            dbc.Col(kpi_card_2, width=2),
            dbc.Col(kpi_card_3, width=2),
            dbc.Col(kpi_card_4, width=2),
            dbc.Col(kpi_card_5, width=2),
            dbc.Col(kpi_card_6, width=2),
            
        ], className = "mb-4"),

    dbc.Row([
        dbc.Col(graph_1, width=4),
        dbc.Col(graph_2, width=4),
        dbc.Col(graph_3, width=4),
    ], className="mb-4"),   

    dbc.Row([
        dbc.Col(graph_4, width=4),
        dbc.Col(graph_5, width=4),
        dbc.Col(graph_6, width=4),
    ], className="mb-4"),
    
], style={"padding": "2rem"})

### 6. Running the Application
Starting the local Dash web server to launch and host the interactive dashboard interface.

In [230]:
if __name__ == "__main__":
    app.run(debug=True, port=8050,jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/
